# Carga de Datos - Riesgo Crediticio

**Autor:** Juan Martín Rossi
**Fecha:** 2026-08-26
**Objetivo:** simular la capa de ingesta de datos que, en un proyecto real, vendría materializada por el Data Warehouse (DWH) o el Datalake de la empresa. Este notebook deja el dataset de créditos otorgados disponible como un archivo plano (`.csv`) dentro de `data/raw/`, listo para que el notebook de análisis exploratorio (`comprension_eda.ipynb`) lo consuma sin depender de Excel.

> **Nota:** en producción esta carga no se haría leyendo un `.xlsx` a mano: el dataset saldría como resultado de otro proceso (una consulta al DWH/Datalake, un job de ETL, una extracción vía API interna, etc.). Acá se usa `Base_de_datos.xlsx` como **dataset de ejemplo no productivo**, entregado para la cursada, solo para poder practicar el flujo ingesta → materialización → EDA.

## Imports y rutas

Se importan únicamente las librerías necesarias para esta etapa: `pandas` para leer y escribir los datos, `numpy` porque se usa como dependencia estándar del stack de análisis del proyecto, y `pathlib` para construir rutas **relativas a la raíz del proyecto**, sin hardcodear rutas absolutas que romperían el notebook en otra máquina.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Si el notebook corre parado en notebooks/ (ejecución normal en Jupyter o
# vía nbconvert), la raíz del proyecto es el directorio padre. Si en algún
# momento se corre con el cwd ya en la raíz, se usa el cwd tal cual.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RUTA_RAW = RAIZ / "data" / "raw"
RUTA_EXCEL = RUTA_RAW / "Base_de_datos.xlsx"
RUTA_CSV = RUTA_RAW / "Base_de_datos.csv"

print("Raíz del proyecto:", RAIZ)
print("Excel de origen:", RUTA_EXCEL)
print("CSV de destino:", RUTA_CSV)

Raíz del proyecto: /Users/juanmartinrossi/Documents/proyecto-final-riesgo-crediticio
Excel de origen: /Users/juanmartinrossi/Documents/proyecto-final-riesgo-crediticio/data/raw/Base_de_datos.xlsx
CSV de destino: /Users/juanmartinrossi/Documents/proyecto-final-riesgo-crediticio/data/raw/Base_de_datos.csv


## Lectura del Excel

Se lee el archivo completo con `dtype=str`: en esta etapa de ingesta el objetivo es solo **materializar** el dataset tal cual llega, sin interpretar tipos. Cualquier casteo (fechas, numéricos, categorías) queda para el notebook de EDA, que es donde esas decisiones se toman con criterio y se documentan.

In [2]:
df_excel = pd.read_excel(RUTA_EXCEL, engine="openpyxl", dtype=str)
df_excel.shape

(10763, 23)

## Verificación de la carga

Antes de dar la carga por buena se revisan: la forma del dataset, una muestra del principio y del final, la lista de columnas y el resumen de `info()`. También se chequea explícitamente si quedaron filas o columnas completamente vacías al final de la hoja, que es un artefacto típico cuando Excel guarda un rango usado más grande que los datos reales.

In [3]:
print("Shape:", df_excel.shape)
df_excel.head(10)

Shape: (10763, 23)


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,3692160,10,42,Independiente,8000000,2500000,341296,88.768094,...,0.0,51258.0,51258.0,0.0,5,0,0,908526,Estable,1
1,4,2025-04-22 09:47:35,840000,6,60,Empleado,3000000,2000000,124876,95.227787,...,0.0,8673.0,8673.0,0.0,0,0,2,939017,Creciente,1
2,9,2026-01-08 12:22:40,5974028.399999999,10,36,Independiente,4036000,829000,529554,47.613894,...,0.0,18702.0,18702.0,0.0,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,1671240,6,48,Empleado,1524547,498000,252420,95.227787,...,0.0,15782.0,15782.0,0.0,3,0,0,1536193,Creciente,1
4,9,2025-04-26 11:24:26,2781636,11,44,Empleado,5000000,4000000,217037,95.227787,...,0.0,204804.0,204804.0,0.0,3,0,1,933473,Creciente,1
5,4,2025-06-10 08:54:01,1031928,12,32,Empleado,2800000,800000,82872,95.227787,...,0.0,24399.0,24399.0,0.0,2,0,8,2808474,Creciente,1
6,4,2025-08-09 13:00:44,3064280.4,6,68,Independiente,1000000,22005000,461231,95.227787,...,0.0,211775.0,211775.0,0.0,6,3,0,969508,Creciente,1
7,4,2025-08-18 12:49:19,3619560,6,31,Empleado,3782303,305000,542079,95.227787,...,0.0,12078.0,12078.0,0.0,0,1,0,3782303,Creciente,1
8,9,2025-05-30 09:11:18,2134827.6,10,31,Empleado,14500000,8000000,181732,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,14007850,Creciente,1
9,4,2024-12-31 14:32:59,8400000,6,45,Empleado,14000000,5000000,1166667,95.227787,...,0.0,125383.0,125383.0,0.0,2,0,0,28889623,Creciente,1


In [4]:
df_excel.tail(5)

,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
10758,9,2025-01-19 16:18:28,2414886,10,29,Independiente,3000000,300000,204819,13.134355,...,172,1112,1112,NaN,0,0,1,NaN,NaN,0
10759,4,2025-01-10 16:40:21,2916000,24,27,Empleado,2500000,400000,127460,55.973342,...,0.0,9771.0,9771.0,0.0,1,0,4,1958333,Creciente,0
10760,4,2025-06-19 14:28:47,4249200,36,24,Empleado,2000000,500000,140042,47.613894,...,0.0,1603.0,1603.0,0.0,1,0,0,998859,Creciente,0
10761,9,2025-03-02 11:53:41,1283307.5999999999,10,26,Empleado,1500000,600000,108958,42.888527,...,0.0,8488.0,8488.0,0.0,2,0,3,NaN,NaN,0
10762,4,2024-12-08 12:46:03,3915000,12,24,Independiente,4000000,2000000,306174,59.32448,...,0.0,5046.0,5046.0,0.0,2,1,1,NaN,NaN,0


In [5]:
print(f"Cantidad de columnas: {len(df_excel.columns)}")
list(df_excel.columns)

Cantidad de columnas: 23


['tipo_credito',
 'fecha_prestamo',
 'capital_prestado',
 'plazo_meses',
 'edad_cliente',
 'tipo_laboral',
 'salario_cliente',
 'total_otros_prestamos',
 'cuota_pactada',
 'puntaje',
 'puntaje_datacredito',
 'cant_creditosvigentes',
 'huella_consulta',
 'saldo_mora',
 'saldo_total',
 'saldo_principal',
 'saldo_mora_codeudor',
 'creditos_sectorFinanciero',
 'creditos_sectorCooperativo',
 'creditos_sectorReal',
 'promedio_ingresos_datacredito',
 'tendencia_ingresos',
 'Pago_atiempo']

In [6]:
df_excel.info()

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   tipo_credito                   10763 non-null  str  
 1   fecha_prestamo                 10763 non-null  str  
 2   capital_prestado               10763 non-null  str  
 3   plazo_meses                    10763 non-null  str  
 4   edad_cliente                   10763 non-null  str  
 5   tipo_laboral                   10763 non-null  str  
 6   salario_cliente                10763 non-null  str  
 7   total_otros_prestamos          10763 non-null  str  
 8   cuota_pactada                  10763 non-null  str  
 9   puntaje                        10763 non-null  str  
 10  puntaje_datacredito            10757 non-null  str  
 11  cant_creditosvigentes          10763 non-null  str  
 12  huella_consulta                10763 non-null  str  
 13  saldo_mora                 

In [7]:
filas_vacias = df_excel.isna().all(axis=1).sum()
columnas_vacias = df_excel.isna().all(axis=0).sum()

print(f"Filas completamente vacías: {filas_vacias}")
print(f"Columnas completamente vacías: {columnas_vacias}")

if filas_vacias == 0 and columnas_vacias == 0:
    print("No se detectó el artefacto típico de Excel (filas/columnas vacías al final de la hoja).")
else:
    print("¡Atención! Hay filas o columnas completamente vacías, revisar antes de continuar.")

Filas completamente vacías: 0
Columnas completamente vacías: 0
No se detectó el artefacto típico de Excel (filas/columnas vacías al final de la hoja).


## Materialización a CSV

Se guarda el dataset como `.csv` dentro de `data/raw/`, con `index=False` (el índice de pandas no es información del negocio), `encoding="utf-8"` (para no perder tildes/ñ) y separador coma. Este archivo es el que va a consumir `comprension_eda.ipynb`.

In [8]:
df_excel.to_csv(RUTA_CSV, index=False, encoding="utf-8", sep=",")
print("CSV guardado en:", RUTA_CSV)

CSV guardado en: /Users/juanmartinrossi/Documents/proyecto-final-riesgo-crediticio/data/raw/Base_de_datos.csv


## Validación de ida y vuelta

Se relee el `.csv` recién escrito y se compara su `shape` contra el del Excel original. Si no coinciden, algo se perdió o se corrompió al escribir (por ejemplo, comas dentro de un campo de texto que rompen el parseo) y la celda tiene que fallar de forma explícita, no seguir en silencio.

In [9]:
df_csv = pd.read_csv(RUTA_CSV, dtype=str, encoding="utf-8", sep=",")

if df_csv.shape != df_excel.shape:
    raise ValueError(
        f"El shape del CSV releído {df_csv.shape} no coincide con el del Excel original "
        f"{df_excel.shape}. Revisar la escritura del CSV antes de continuar."
    )

print(f"Validación OK: el CSV releído tiene el mismo shape que el Excel original: {df_csv.shape}")

Validación OK: el CSV releído tiene el mismo shape que el Excel original: (10763, 23)


## Cierre

- El dataset quedó materializado en `data/raw/Base_de_datos.csv`, con **10.763 filas** y **23 columnas**, todas como texto (sin castear todavía).
- La validación de ida y vuelta confirmó que no se perdió información al escribir el CSV.
- El próximo paso es `comprension_eda.ipynb`, donde se hace la exploración inicial, se sanean los tipos de cada columna y se unifican los distintos formatos de valores faltantes.